In [ ]:
import pandas as pd
import numpy as np
import os
import itertools as it
from snp_analysis_tools_sherlock import *
from coalescence_analysis_tools import *
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')

In [ ]:
fname = '~/git/coalescence-pilot-mgx/workflow/out/midas2_output/old_species/species/metadata.tsv'
df_metadata= pd.read_csv(fname, delimiter = '\t')
df_metadata

def transform_df(df_abundance):
    df_abundance['Lineage'] = df_abundance['species_id'].transform(lambda x: df_metadata.loc[df_metadata['species_id'] == x,'Lineage'].values[0])
    df_abundance['species'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-1])
    df_abundance['genus'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-2])
    df_abundance['family'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-3])
    df_abundance['phyla'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[1])
    return df_abundance

df_metadata  =transform_df(df_metadata)

df_abundance = pd.read_csv('e003_round4_assembly_abundances.csv')
df_abundance.loc[df_abundance['exp_type']=='e003GlycerolRevival','passage']=df_abundance.loc[df_abundance['exp_type']=='e003GlycerolRevival','passage']+5
#df_abundance  =transform_df(df_abundance)
df_abundance.head()

In [ ]:
df_meso_alts=df_abundance.groupby(['sample','exp_type','media',
                      'passage','comm','subject','mesocosm','type_mesocosm']).sum().reset_index()
df_meso_alts['relative_abundance']=1-df_meso_alts['relative_abundance'].sum()
df_meso_alts=df_meso_alts.loc[df_meso_alts['relative_abundance']>0,:]
df_meso_alts['genus']='g__missing'
df_meso_alts['family']='f__missing'
df_meso_alts['phyla']='p__missing'
df_meso_alts['species']='s__missing'
df_meso_alts['Lineage']='missing'
df_meso_alts['species_id']=0
df_abundance # = pd.concat([df_abundance,df_meso_alts])

In [ ]:
def adjust_df(meso_df, good_families ):
    for i,fam in enumerate(good_families):
        meso_df.loc[(meso_df['family']==fam)*(meso_df['relative_abundance']<1e-2),'species_id'] = i
    meso_df = meso_df.groupby(['family','sample','passage','species_id',]).sum().reset_index()
                               
    for sample in meso_df['sample'].unique():
        full= meso_df.loc[meso_df['sample']==sample,'relative_abundance'].sum()
        if full>0:
            #print(sample,full)
            meso_df.loc[meso_df['sample']==sample,'relative_abundance']=  meso_df.loc[meso_df['sample']==sample,'relative_abundance']/full
    
    return meso_df
    

In [ ]:
meso_to_look_at = 'B5-AF-mBHI'
df_meso=df_abundance.loc[df_abundance['mesocosm']==meso_to_look_at,:]
sub = meso_to_look_at.split('-')[1]
in_sample = df_abundance.loc[df_abundance['mesocosm']==f'fecal-{sub}-fecal',:]
df_meso = pd.concat([df_meso,in_sample])
df_meso['passage_plot']=df_meso['passage'].astype(str)
cmap_family= {'f__Bacteroidaceae': '#8dd3c7',
 'f__Enterobacteriaceae': '#ffffb3',
 'f__Porphyromonadaceae': '#bebada',
 'f__Peptoniphilaceae': '#fb8072',
 'f__Oscillospiraceae': '#80b1d3',
 'f__Enterococcaceae': '#fdb462',
 'f__Tannerellaceae': '#b3de69',
 'f__Lachnospiraceae': '#fccde5',
 'f__Veillonellaceae': '#bc80bd',
 'f__Peptostreptococcaceae': '#ccebc5',
 'f__Acidaminococcaceae': '#ffed6f',
 'f__other': '#d9d9d9'}
good_families = list(cmap_family.keys())
df_meso = adjust_df(df_meso, good_families)
df_meso['family_plot']=df_meso['family'].copy()
df_meso.loc[~df_meso['family'].isin(good_families),'family_plot']='f__other'
df_meso=df_meso.sort_values(by='family_plot')
df_meso_small =df_meso.loc[df_meso['passage'].isin([0,1,2,3,4,5]),:].sort_values(by='family_plot')
bars2 = hv.Bars(df_meso_small, kdims=[hv.Dimension('passage', values=[0,1,2,3,4,5]), 'species_id',],
               vdims = ['relative_abundance','family_plot',])

bars2=bars2.opts(width=600, height=400).opts(stacked=True,#alpha='relative_abundance',
                                      color='family_plot',
                                      cmap=cmap_family,
                                      alpha=1.,
                                             ylim = (0,1),
                                             bar_width = 2,
                                      xlabel='Passage',
                                      ylabel='Relative Abundance',
                                        show_legend=True,legend_position='right',
                                         )#.sort(by='passage_plot')

bars2

In [ ]:
sp_good = df_meso_small['species_id'].unique()
df_meso_small_adjust = df_meso_small.copy()
new_df = []
for passage in df_meso_small['passage'].unique():
    df_meso_smallg = df_meso_small.loc[df_meso_small['passage']==passage,:].copy()
    df_meso_smallg['passage']=passage+.2
    new_df.append(df_meso_smallg)
    df_meso_smallb = df_meso_small.loc[df_meso_small['passage']==passage,:].copy()
    df_meso_smallb['passage']=passage-.2
    new_df.append(df_meso_smallb)
new_df = pd.concat(new_df)
df_meso_small_big = pd.concat([new_df, df_meso_small])
overlay = hv.Overlay([hv.Area(df_meso_small_big.loc[df_meso_small_big['species_id']==sp,:].sort_values(by='passage'), kdims=[hv.Dimension('passage',values=[0,1,2,3,4,5],)],
                              vdims=['relative_abundance','family_plot']).opts(alpha=.5,line_color='grey',
                                                                               color=cmap_family[df_meso_small.loc[df_meso_small['species_id']==sp,'family_plot'].values[0]]) for sp in sp_good])



p=hv.Area.stack(overlay)
(p*bars2).opts(xlim=(-0.25,5.5),ylim=(0,1))

In [ ]:
sums = df_meso.groupby(['sample']).sum(numeric_only=True)
good_samples = sums.loc[sums['relative_abundance']>0.,:].index.values
good_samples

In [ ]:
def adjust_df(meso_df, good_families ):
    for i,fam in enumerate(good_families):
        meso_df.loc[(meso_df['family']==fam)*(meso_df['relative_abundance']<1e-2),'species_id'] = i
    meso_df = meso_df.groupby(['family','sample','passage','species_id',]).sum().reset_index()
                               
    for sample in meso_df['sample'].unique():
        full= meso_df.loc[meso_df['sample']==sample,'relative_abundance'].sum()
        if full>0:
            #print(sample,full)
            meso_df.loc[meso_df['sample']==sample,'relative_abundance']=  meso_df.loc[meso_df['sample']==sample,'relative_abundance']/full
    
    return meso_df

In [ ]:
def make_bar_plot_assembly(meso_to_look_at, df_abundance,add=.2):
    df_meso=df_abundance.loc[df_abundance['mesocosm']==meso_to_look_at,:]
    sub = meso_to_look_at.split('-')[1]
    in_sample = df_abundance.loc[df_abundance['mesocosm']==f'fecal-{sub}-fecal',:]
    df_meso = pd.concat([df_meso,in_sample])
    df_meso['passage_plot']=df_meso['passage'].astype(str)
    df_meso['family'] = df_meso['family'].transform(lambda x: x.split('f__')[-1])
    sums = df_meso.groupby(['sample']).sum(numeric_only=True)
    good_samples = sums.loc[sums['relative_abundance']>0.,:].index.values
    df_meso=df_meso.loc[df_meso['sample'].isin(good_samples),:]
    cmap_family= {'Bacteroidaceae': '#8dd3c7',
    'Enterobacteriaceae': '#ffffb3',
    'Porphyromonadaceae': '#bebada',
    'Peptoniphilaceae': '#fb8072',
    'Oscillospiraceae': '#80b1d3',
    'Enterococcaceae': '#fdb462',
    'Tannerellaceae': '#b3de69',
    'Lachnospiraceae': '#fccde5',
    'Veillonellaceae': '#bc80bd',
    'Peptostreptococcaceae': '#ccebc5',
    'Acidaminococcaceae': '#ffed6f',
    'other': '#d9d9d9'}
    good_families = list(cmap_family.keys())
    df_meso.loc[~df_meso['family'].isin(good_families),'family']='other'
    df_meso = adjust_df(df_meso, good_families)
    df_meso['family_plot']=df_meso['family'].copy()
    #df_meso.loc[~df_meso['family'].isin(good_families),'family_plot']='other'
    df_meso=df_meso.sort_values(by='family_plot')
    df_meso_small =df_meso.loc[df_meso['passage'].isin([0,1,2,3,4,5]),:].sort_values(by='family_plot')
    bars2 = hv.Bars(df_meso_small, kdims=[hv.Dimension('passage', values=[0,1,2,3,4,5]), 'species_id',],
                vdims = ['relative_abundance','family_plot',])

    bars2=bars2.opts(width=600, height=400).opts(stacked=True,#alpha='relative_abundance',
                                        color='family_plot',
                                        cmap=cmap_family,
                                        alpha=1,
                                                ylim = (0,1),
                                                bar_width = .8,
                                        xlabel='Passage',
                                        ylabel='Relative Abundance',
                                            show_legend=True,legend_position='right',
                                            )#.sort(by='passage_plot')   
    sp_good = df_meso_small['species_id'].unique()
    df_meso_small_adjust = df_meso_small.copy()
    new_df = []
    for passage in df_meso_small['passage'].unique():
        df_meso_smallg = df_meso_small.loc[df_meso_small['passage']==passage,:].copy()
        df_meso_smallg['passage']=passage+add
        new_df.append(df_meso_smallg)
        df_meso_smallb = df_meso_small.loc[df_meso_small['passage']==passage,:].copy()
        df_meso_smallb['passage']=passage-add
        new_df.append(df_meso_smallb)
    new_df = pd.concat(new_df)
    df_meso_small_big = pd.concat([new_df, df_meso_small])
    overlay = hv.Overlay([hv.Area(df_meso_small_big.loc[df_meso_small_big['species_id']==sp,:].sort_values(by='passage'), kdims=[hv.Dimension('passage',values=[0,1,2,3,4,5],)],
                                vdims=['relative_abundance','family_plot']).opts(alpha=.5,line_color='grey',
                                                                                color=cmap_family[df_meso_small.loc[df_meso_small['species_id']==sp,'family_plot'].values[0]]) for sp in sp_good])

    p=hv.Area.stack(overlay)
    return (p*bars2).opts(xlim=(-0.25,5.5),ylim=(0,1))



In [ ]:
p = make_bar_plot_assembly(meso_to_look_at, df_abundance,add=.2)

In [ ]:
p

In [ ]:
meso_to_look_at = 'C11-AF-mGAM'

df_meso=df_abundance.loc[df_abundance['mesocosm']==meso_to_look_at,:]
in_sample = df_abundance.loc[df_abundance['mesocosm']=='fecal-AF-fecal',:]
df_meso = pd.concat([df_meso,in_sample])
df_meso['family_sp']=df_meso['family']+ '-' + df_meso['species']
df_meso['passage'].unique()
#df_meso['species_id']=df_meso['species_id'].astype(str)
df_meso['passage_plot']=df_meso['passage'].astype(str)
#df_meso=df_meso.sort_values(by='passage_plot')
df_meso['is_important']=0.1
df_meso.loc[df_meso['species_id']==101346,'is_important']=1.
#df_meso=df_meso.loc[df_meso['sample']!='Assembly-G12-fecal-AA-fecal-0_S703',:]
cmap_family= {'f__Bacteroidaceae': '#8dd3c7',
 'f__Enterobacteriaceae': '#ffffb3',
 'f__Porphyromonadaceae': '#bebada',
 'f__Peptoniphilaceae': '#fb8072',
 'f__Oscillospiraceae': '#80b1d3',
 'f__Enterococcaceae': '#fdb462',
 'f__Tannerellaceae': '#b3de69',
 'f__Lachnospiraceae': '#fccde5',
 'f__Veillonellaceae': '#bc80bd',
 'f__Peptostreptococcaceae': '#ccebc5',
 'f__Acidaminococcaceae': '#ffed6f',
 'f__other': '#d9d9d9'}
good_families = list(cmap_family.keys())
good_families

df_meso.loc[df_meso['relative_abundance']<1e-2,'relative_abundance']=0
#print(df_meso['family'])
df_meso = adjust_df(df_meso)
#print(df_meso['family'])
df_meso['family_plot']=df_meso['family'].copy()

df_meso.loc[~df_meso['family'].isin(good_families),'family_plot']='f__other'
df_meso=df_meso.sort_values(by='family_plot')
#df_mesog=df_meso#.loc[df_meso['is_important']==1.,:]
df_meso_small =df_meso.loc[df_meso['passage'].isin([0,5]),:].sort_values(by='family_plot')
df_meso_small['passage']=df_meso_small['passage'].astype(int)
bars = hv.Bars(df_meso_small, kdims=[hv.Dimension('passage', values=[0,5]), 'species_id',],
               vdims = ['relative_abundance','family_plot',])

bars=bars.opts(width=600, height=400).opts(stacked=True,#alpha='relative_abundance',
                                      color='family_plot',
                                          # size=10,
                                           bar_width=4,
                                      cmap=cmap_family,
                                      alpha=1.,
                                      xlabel='Passage',
                                      ylabel='Relative Abundance',
                                        show_legend=True,legend_position='right',
                                         )#.sort(by='passage_plot')

bars

In [ ]:
sp_good = df_meso_small['species_id'].unique()
df_meso_small_adjust = df_meso_small.copy()
new_df = []
for passage in df_meso_small['passage'].unique():
    df_meso_smallg = df_meso_small.loc[df_meso_small['passage']==passage,:].copy()
    df_meso_smallg['passage']=passage+.25
    new_df.append(df_meso_smallg)
    df_meso_smallb = df_meso_small.loc[df_meso_small['passage']==passage,:].copy()
    df_meso_smallb['passage']=passage-.25
    new_df.append(df_meso_smallb)
new_df = pd.concat(new_df)
df_meso_small_big = pd.concat([new_df, df_meso_small])
overlay = hv.Overlay([hv.Area(df_meso_small_big.loc[df_meso_small_big['species_id']==sp,:].sort_values(by='passage'), kdims=[hv.Dimension('passage',values=[0,5],)],
                              vdims=['relative_abundance','family_plot']).opts(alpha=.5,line_color='grey',
                                                                               color=cmap_family[df_meso_small.loc[df_meso_small['species_id']==sp,'family_plot'].values[0]]) for sp in sp_good])



p=hv.Area.stack(overlay)
(p*bars).opts(xlim=(-0.25,5.5),ylim=(0,1))

In [ ]:
#df_meso=df_meso.sort_values(by='family_plot')
#df_meso_small =df_meso.loc[df_meso['passage'].isin([0,1,2,3,4,5]),:]
sp_good = df_meso_small['species_id'].unique()
df_meso_small_adjust = df_meso_small.copy()
new_df = []
for passage in df_meso_small['passage'].unique():
    df_meso_smallg = df_meso_small.loc[df_meso_small['passage']==passage,:].copy()
    df_meso_smallg['passage']=passage+.2
    new_df.append(df_meso_smallg)
    df_meso_smallb = df_meso_small.loc[df_meso_small['passage']==passage,:].copy()
    df_meso_smallb['passage']=passage-.2
    new_df.append(df_meso_smallb)
new_df = pd.concat(new_df)
df_meso_small_big = pd.concat([new_df, df_meso_small])
overlay = hv.Overlay([hv.Area(df_meso_small_big.loc[df_meso_small_big['species_id']==sp,:].sort_values(by='passage'), kdims=[hv.Dimension('passage',values=[0,1,2,3,4,5],)],
                              vdims=['relative_abundance','family_plot']).opts(alpha=.1,line_color='grey',
                                                                               color=cmap_family[df_meso_small.loc[df_meso_small['species_id']==sp,'family_plot'].values[0]]) for sp in sp_good])



p=hv.Area.stack(overlay)
(p*bars).opts(xlim=(-0.25,5.5),ylim=(0,1))

In [ ]:
(p*bars).opts(xlim=(-0.25,5.5),ylim=(0,1))